# 04 — ANOVA Decomposition

Reproduces paper **Appendix Table 6** (η²_prompt, η²_question,
η²_interaction averaged across layers and concepts, per model).

Implementation in `grace/diagnostics/anova.py`. Reads activations
directly — no judging or steering needed.

> **Note:** This notebook loads pre-computed per-concept JSON summaries
> from `results/anova/` when available (fast path). If a cached result is
> missing, it falls back to recomputing from raw activations (~122 GB).
> To regenerate the cached JSONs from scratch, run `scripts/09_run_anova.py`
> after the full activation-extraction pipeline (see README).

In [3]:
import os
from pathlib import Path
import json
import pandas as pd

# Ensure working directory is the repo root so relative paths resolve correctly.
_repo_root = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd()
if (_repo_root / 'grace').is_dir():
    os.chdir(_repo_root)
elif (_repo_root.parent / 'grace').is_dir():
    os.chdir(_repo_root.parent)

from grace.diagnostics.anova import anova_decomposition

MODELS = {
    'gemma2-2b': 'google/gemma-2-2b-it',
    'gemma3-27b': 'google/gemma-3-27b-it',
    'llama3-70b': 'meta-llama/Llama-3.3-70B-Instruct',
}
CONCEPTS = sorted(p.stem for p in Path('concepts/gpt-5/extract').glob('*.json'))
OUT = Path('results/anova'); OUT.mkdir(parents=True, exist_ok=True)

print(f"CWD: {Path.cwd()}")
print(f"Concepts: {len(CONCEPTS)}")

CWD: /home/jtr2875/GRACE
Concepts: 20


In [4]:
rows = []
for tag, mname in MODELS.items():
    model_dir = mname.split('/')[-1]
    for c in CONCEPTS:
        cached = OUT / model_dir / f'{c}.json'
        # Prefer cached JSON (fast); recompute from activations only if missing.
        if cached.exists():
            with open(cached) as f:
                per_layer = json.load(f)['per_layer']
            vals = list(per_layer.values())
            d = {
                'mean_eta_sq_prompt': sum(v['eta_sq_prompt'] for v in vals) / len(vals),
                'mean_eta_sq_question': sum(v['eta_sq_question'] for v in vals) / len(vals),
                'mean_eta_sq_interaction': sum(v['eta_sq_interaction'] for v in vals) / len(vals),
            }
        else:
            try:
                d = anova_decomposition(mname, c, activation_type='response_avg')
            except FileNotFoundError:
                continue
        rows.append({
            'model': tag, 'concept': c,
            'eta_prompt': d['mean_eta_sq_prompt'],
            'eta_question': d['mean_eta_sq_question'],
            'eta_interaction': d['mean_eta_sq_interaction'],
        })

if not rows:
    print("No ANOVA data found. Ensure activations or cached results/anova/ JSONs are available.")

df = pd.DataFrame(rows)
summary = df.groupby('model')[['eta_prompt', 'eta_question', 'eta_interaction']].mean().round(3)
summary.loc['All models'] = summary.mean()
summary

,eta_prompt,eta_question,eta_interaction
model,,,
gemma2-2b,0.144000,0.430000,0.426
gemma3-27b,0.178000,0.379000,0.443
llama3-70b,0.156000,0.464000,0.379
All models,0.159333,0.424333,0.416
